In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Load the CSV file
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

import warnings



Q1_path = os.path.join(path, 'Q1_data.csv')
df_data = pd.read_csv(Q1_path)

print(f"Shape: {df_data.shape}")


In [ ]:
df_data.head()

In [ ]:
df_data.info()

In [ ]:
df_data.describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_data['Delivery_Time'].dropna(), bins=30, edgecolor='black', color='orange')
plt.title('delivery_time')
plt.xlabel('delivery')
plt.ylabel('Time')
plt.show()

In [ ]:


cols = ['Order_ID' ,'Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df_data[cols].copy()



print(f"Before: {df_clean.shape}")
df_clean = df_clean.drop(columns=['Order_ID'])
print(f"After dropping Order_ID : {df_clean.shape}")





In [ ]:
# Drop rows where the target variable
initial_shape = df_clean.shape[0]
df_clean.dropna(subset=['Delivery_Time'], inplace=True)
print(f"Dropped {initial_shape - df_clean.shape[0]} rows with missing Delivery_Time.")

# missing
if 'Courier_Experience_yrs' in df_clean.columns:
    median_experience = df_clean['Courier_Experience_yrs'].median()
    df_clean['Courier_Experience_yrs'].fillna(median_experience, inplace=True)
    print(f"Imputed missing Courier_Experience_yrs with median: {median_experience}")

# Fill missing categorical values with 'unknown'
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    if col in df_clean.columns:
        df_clean[col].fillna('unknown', inplace=True)
        print(f"Filled missing values in {col} with 'unknown'.")

print("Missing values remaining after handling:", df_clean.isnull().sum().sum())

# Check and remove duplicates
def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Do we have duplicate samples?
def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Identify categorical columns
categorical_cols = df_clean.select_dtypes(include='object').columns


df_clean = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

df_clean.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time", errors='ignore')

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop("Delivery_Time", axis=1)
y = df_clean['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Initialize the model
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []



for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Scale features within the fold
    scaler_fold = StandardScaler()
    X_fold_train_scaled = scaler_fold.fit_transform(X_fold_train)
    X_fold_val_scaled = scaler_fold.transform(X_fold_val)

    # Train and predict
    model.fit(X_fold_train_scaled, y_fold_train)
    y_fold_pred = model.predict(X_fold_val_scaled)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)


print(f"MAE:  {mae_scores.mean():,.2f}")



final_scaler = StandardScaler()
X_train_final_scaled = final_scaler.fit_transform(X_train)
final_model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
final_model.fit(X_train_final_scaled, y_train)
print("Final model trained ")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

feature_importances = final_model.feature_importances_
feature_names = X_train.columns


importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})

importance_df = importance_df.sort_values(by='Importance', ascending=False)


plt.figure(figsize=(12, 8))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='blue')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature')
plt.show()

In [ ]:
import matplotlib.pyplot as plt


X_test_scaled = final_scaler.transform(X_test)

y_pred = final_model.predict(X_test_scaled)

plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=30, edgecolor='black', color='blue')
plt.title('Times')
plt.xlabel('Predicted Delivery')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: